In [1]:
from openai import OpenAI
import openai
import os
import fitz
import faiss
import numpy as np


api_key = os.getenv("OPENAI_API_KEY")
openai.api_key = api_key
client = OpenAI()

In [2]:
def extract_text_chunks( pdf_path, chunk_size=500 ):
    text_chunks =[]
    with fitz.open( pdf_path ) as pdf:
        for page in pdf:
            text = page.get_text()
            text_chunks.extend( [ text[i:i+chunk_size] for i in range(0, len(text), chunk_size) ]  )
    return text_chunks

def get_embedding( text ):
    response = client.embeddings.create( input=text, model="text-embedding-ada-002")
    return np.array( response.data[0].embedding ,dtype=np.float32 )

def store_embeddings_in_faiss( text_chunks):
    # 생성된 벡터들을 수직으로 쌓아 하나의 2차원배열을 만든다
    # (청크개수, 임베딩 차원 ) 배열
    embeddings = np.vstack(  [get_embedding(chunk) for chunk in text_chunks ] )
    index = faiss.IndexFlatL2( embeddings.shape[1] )
    index.add( embeddings  ) # embedding 벡터db저장
    return index

# 질문에 응답함수
def answer_question( index, text_chunks, question):
    q_emb = get_embedding( question).reshape(1,-1)
    _ , top_index = index.search( q_emb ,k=5 )
    context = " ".join( [text_chunks[i] for i in top_index[0] ] )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{'role':'user',
            'content':f'Context:{context}\n\nQuestion:{question}\nAnswer:'}]
    )
    return response.choices[0].message.content

text_chunks = extract_text_chunks( 'pdf/The_Adventures_of_Tom_Sawyer.pdf')

index = store_embeddings_in_faiss( text_chunks)
question = "Tell me about the characters"
rst = answer_question(index, text_chunks, question)
print(rst)

1. Tom: Tom is a young boy who is feeling sad and doesn't want to go to school. He is adventurous and impulsive, as shown by his decision to run away with Joe and Huck. He also witnesses a murder in the graveyard and ends up being mistaken for dead. Tom is resourceful and clever, as he manages to escape and return home safely.

2. Joe Harper: Joe is another young boy who is feeling sad because his mother is angry with him. He agrees to run away with Tom and Huck to have an adventure. Joe ultimately decides to go back home with Tom and Huck.

3. Huck Finn: Huck is a friend of Tom and Joe who is also feeling sad. He joins Tom and Joe in running away and suggests going across the river for an adventure. Huck is adventurous and enjoys taking risks.

4. Injun Joe: Injun Joe is a character who is involved in a murder in the graveyard. Tom witnesses him killing the doctor and wrongly believes that he is dead. Injun Joe is portrayed as cunning and dangerous.

5. Aunt Polly: Aunt Polly is Tom's

1. pdf 텍스트 읽어오기
2. 500개글자씩 자르기 (일반적으로 500~1000개 사이)
3. 자른 글자를 임베딩하기
4. 벡터 db에 저장하기
5. 질문을 임베딩
6. 유사도검색
7. 유사도 높은 실제 문장을 하나의 글자로 만든다 
- context:유사도검색된 문장 
- question:질문
- answer:
- 프롬프팅
8. gpt가 질문에 대한 답변을 context 에 기반하여 답변한다

# 🌐 OpenAI `text-embedding-ada-002` 모델 특성

`text-embedding-ada-002`는 단순히 영어만 지원하는 임베딩 모델이 아닙니다.  
한국어, 영어, 일본어, 중국어 등 **다국어**를 다음과 같은 방식으로 처리합니다:

---

## 공통 의미 공간 (Shared Semantic Space)

- 다양한 언어를 **공통된 의미 공간** 안에서 임베딩합니다.
- 즉, **언어가 달라도 의미가 비슷하면 벡터가 유사**하게 나타납니다.

---

## 📘 예시 설명

> `"The Adventures of Tom Sawyer"`라는 영어 텍스트를 임베딩하면  
> 해당 문장의 의미가 하나의 **벡터**로 표현됩니다.

> 그리고 한글 질문 `"줄거리 요약해줘"`도  
> 그 의미에 따라 **유사한 벡터**로 변환됩니다.

---

## ✅ 핵심 요약

- 💬 **다국어 입력** 지원  
- 📐 **언어 간 의미 기반 비교 가능**  
- 🔍 **RAG 및 검색 시스템에 매우 유용**

> **"의미가 비슷하면 언어가 달라도 벡터가 가까워진다."**

---

## 🔗 활용 예시

| 질문 텍스트                          | 의미 벡터 예시                |
|-------------------------------------|-------------------------------|
| `Summarize the plot`     | 📐 `0.32, -0.91, 0.10, ...`   |
| `줄거리 요약해줘`                   | 📐 `0.31, -0.90, 0.11, ...`   |
| `あらすじを要約して` | 📐 `0.33, -0.92, 0.09, ...`   |

→ **서로 다른 언어지만, 유사한 의미 → 유사한 벡터**

---

📌 이처럼 `text-embedding-ada-002`는  
**의미 기반 검색, 분류, 유사도 판단** 등에 매우 강력한 성능을 발휘합니다.
